# Task 2: Build Time Series Forecasting Models

Compare ARIMA/SARIMA and LSTM on Tesla (TSLA) price forecasting.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from src.data_loader import combine_close_prices, fetch_all_assets
from src.forecasting import (
    build_lstm_model,
    chronological_split,
    compute_metrics,
    create_lstm_sequences,
    fit_auto_arima,
    forecast_arima,
    metrics_to_frame,
    predict_lstm,
    train_lstm,
)
from src.preprocessing import clean_asset_frame

LOOKBACK = 60
TRAIN_END = '2024-12-31'

In [ ]:
asset_data = {t: clean_asset_frame(df) for t, df in fetch_all_assets().items()}
tsla = combine_close_prices(asset_data)['TSLA'].dropna()
train, test = chronological_split(tsla, TRAIN_END)
print(f'Train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} days)')
print(f'Test:  {test.index.min().date()} to {test.index.max().date()} ({len(test)} days)')

## ARIMA / SARIMA

In [ ]:
arima_model = fit_auto_arima(train, seasonal=True, m=5)
print(arima_model.summary())
arima_pred = forecast_arima(arima_model, len(test))
arima_metrics = compute_metrics(test.values, arima_pred)
metrics_to_frame('ARIMA/SARIMA', arima_metrics)

## LSTM

In [ ]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train.values.reshape(-1, 1))
full_scaled = scaler.transform(tsla.values.reshape(-1, 1))

x_all, y_all = create_lstm_sequences(full_scaled, LOOKBACK)
train_size = len(train) - LOOKBACK
x_train, y_train = x_all[:train_size], y_all[:train_size]
x_test = x_all[train_size:train_size + len(test)]

lstm_model = build_lstm_model(LOOKBACK, units=50, learning_rate=0.001)
train_lstm(lstm_model, x_train.reshape(-1, LOOKBACK, 1), y_train, epochs=25, batch_size=32)

lstm_pred_scaled = predict_lstm(lstm_model, x_test.reshape(-1, LOOKBACK, 1))
lstm_pred = scaler.inverse_transform(lstm_pred_scaled.reshape(-1, 1)).flatten()
lstm_metrics = compute_metrics(test.values[:len(lstm_pred)], lstm_pred)
metrics_to_frame('LSTM', lstm_metrics)

## Model Comparison

In [ ]:
comparison = pd.concat([
    metrics_to_frame('ARIMA/SARIMA', arima_metrics),
    metrics_to_frame('LSTM', lstm_metrics),
], ignore_index=True)
comparison

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test.index, test.values, label='Actual', linewidth=2)
plt.plot(test.index, arima_pred, label='ARIMA/SARIMA', alpha=0.8)
plt.plot(test.index[:len(lstm_pred)], lstm_pred, label='LSTM', alpha=0.8)
plt.title('TSLA Test Set Forecast Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Selection rationale:** Compare RMSE and MAPE. ARIMA is interpretable and fast; LSTM can capture nonlinear patterns but needs more data and tuning. The model with lower test RMSE is preferred for Task 3.